In [ ]:
!pip install numpy==1.26.4

In [ ]:
!pip install scikit-surprise

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
netflix_dataset=pd.read_csv("/content/combined_data_1.txt", header=None,
                            names=["Cust_id", "Ratings"], usecols=[0,1])


In [ ]:
netflix_dataset

In [ ]:
movies_id=[]
movies_count=0

for i in netflix_dataset["Cust_id"]:
  if ":" in i:
    movies_count+=1
    id=int(i.replace(":",""))
  movies_id.append(id)
print("Total Number of Movies are: ",movies_count)

Total Number of Movies are:  4499


In [ ]:
netflix_dataset["Movie_id"]=movies_id

In [ ]:
netflix_dataset.dropna(inplace=True)
netflix_dataset

In [ ]:
netflix_dataset.dtypes

In [ ]:
netflix_dataset["Cust_id"]=netflix_dataset["Cust_id"].astype(int)
netflix_dataset.dtypes

## EDA ##

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1,1, figsize=(10, 6))

sns.countplot(x='Ratings', data=netflix_dataset, ax=axes, palette='viridis',hue="Ratings", legend=False)
axes.set_title('Distribution of Netflix Ratings', fontsize=14)
axes.set_xlabel('Rating (Stars)', fontsize=12)
axes.set_ylabel('Count (in Millions)', fontsize=12)
plt.tight_layout()
plt.show()



In [ ]:

top_movies = netflix_dataset['Movie_id'].value_counts().head(10).reset_index()
top_movies.columns = ['Movie_id', 'Rating_Count']

top_movies_with_titles = top_movies.merge(df_names, left_on='Movie_id', right_on='movieId')
fig, axes = plt.subplots(1,1, figsize=(10, 6))
sns.barplot(x='Rating_Count', y='title', data=top_movies_with_titles, ax=axes, hue='title', palette='magma', legend=False)
axes.set_title('Top 10 Most Popular Movies (By Rating Volume)', fontsize=14)
axes.set_xlabel('Number of Ratings', fontsize=12)
axes.set_ylabel('Movie Title', fontsize=12)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

user_counts = netflix_dataset['Cust_id'].value_counts()

sns.histplot(user_counts, bins=50, kde=True, color='crimson', log_scale=True)
plt.axvline(x=cust_benchmark, color='blue', linestyle='--', label=f' Benchmark ({cust_benchmark})')

plt.title('User Activity Distribution (Log Scale)', fontsize=14)
plt.xlabel('Number of Ratings Given by an Individual User', fontsize=12)
plt.ylabel('Number of Users', fontsize=12)
plt.legend()
plt.show()

In [ ]:
genres_split = df_names['genres'].str.split('|').explode()

plt.figure(figsize=(12, 6))
sns.countplot(y=genres_split, order=genres_split.value_counts().index, palette='coolwarm')

plt.title('Distribution of Movie Genres in Dataset', fontsize=14)
plt.xlabel('Number of Movies', fontsize=12)
plt.ylabel('Genre', fontsize=12)
plt.show()

## Data Filtering ##

In [ ]:
cust_data=netflix_dataset["Cust_id"].value_counts()
cust_data

In [ ]:
cust_benchmark=round(cust_data.quantile(0.6))
cust_benchmark

36

In [ ]:
movies_data=netflix_dataset["Movie_id"].value_counts()
movies_data

In [ ]:
movies_benchmark=round(movies_data.quantile(0.6))
movies_benchmark

908

In [ ]:
drop_cust=cust_data[cust_data<cust_benchmark].index
drop_cust

Index([2194851,  600295, 1739398, 1157368,  532108, 2157249,  256134,  640441,
       1272324, 1346990,
       ...
       1969065,  899932,  611596, 2147176,  811650, 1300341, 2550360,   11848,
        930788,  594210],
      dtype='int64', name='Cust_id', length=282042)

In [ ]:
drop_movies=movies_data[movies_data<movies_benchmark].index
drop_movies

Index([1598, 1733, 1647, 4099, 1616, 1446,  263, 4259,  160, 1988,
       ...
       1858, 4035, 3693, 2805,  820, 4294,  915, 3656, 4338, 4362],
      dtype='int64', name='Movie_id', length=2699)

In [ ]:
netflix_dataset=netflix_dataset[~netflix_dataset["Cust_id"].isin(drop_cust) ]
netflix_dataset

In [ ]:
netflix_dataset=netflix_dataset[~netflix_dataset["Movie_id"].isin(drop_movies)]
netflix_dataset

In [ ]:
df_names=pd.read_csv("/content/movies.csv", usecols=[0,1,2])
df_names

## Modelling  ##

In [ ]:
from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate

In [ ]:
reader=Reader(rating_scale=(1,5))

In [ ]:
data=Dataset.load_from_df(netflix_dataset[["Cust_id","Movie_id","Ratings"]],reader)
data

In [ ]:
model=SVD()

In [ ]:
cross_validate(model,data,measures=["RMSE"],cv=3)

{'test_rmse': array([1.01891151, 1.01748229, 1.01920617]),
 'fit_time': (1.46238112449646, 2.176429271697998, 1.175156831741333),
 'test_time': (0.32086968421936035, 0.5260283946990967, 0.17464137077331543)}

In [ ]:
traindata=data.build_full_trainset()

In [ ]:
model.fit(traindata)

In [ ]:
#model.predict(823519,3421).est

In [ ]:
user_rating=netflix_dataset[netflix_dataset["Cust_id"]==823519]
user_rating

In [ ]:
df_names = pd.read_csv("/content/movies.csv", usecols=[0, 1, 2], header=0)
df_names.columns = ['movieId', 'title', 'genres']

In [ ]:
user_id=823519

df_names['movieId'] = df_names['movieId'].astype(int)

# 2. Get the IDs of movies this user has already rated
rated_movies = user_rating['Movie_id'].unique()

# 3. Filter out movies the user has already seen AND movies from your drop_movies list
unrated_movies = df_names[
    (~df_names["movieId"].isin(rated_movies)) &
    (~df_names["movieId"].isin(drop_movies))
].copy()

# 4. Predict ratings for these unrated movies using your trained SVD model
unrated_movies["Estimated_score"] = unrated_movies['movieId'].apply(
    lambda x: model.predict(user_id, int(x)).est
)

# 5. Sort by highest score and get the top 10 recommendations
recommendations = unrated_movies.sort_values('Estimated_score', ascending=False).head(10)

# 6. Display the final recommendation dataframe
recommendations[["movieId", "title", "genres", "Estimated_score"]]